In [1]:
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

import lsst.geom
from lsst.daf.butler import Butler
from lsst.rsp import get_tap_service

In [2]:
rsp_tap = get_tap_service("tap")
assert rsp_tap is not None

In [3]:
butler = Butler('dp1', collections='LSSTComCam/DP1')
assert butler is not None

In [4]:
ra, dec = 106.3, -10.40
my_band = 'r'

In [5]:
# Step 1: Query Butler for a visit_image by sky position and band
visitimage_refs = butler.query_datasets(
        'visit_image',
        where="visit.region OVERLAPS POINT(ra, dec) "
              f"and band='{my_band}'",
        bind={'ra': ra, 'dec': dec},
        order_by=['visit', 'detector'],
        with_dimension_records=True,
        limit=1
)

print(f"Found {len(visitimage_refs)} visit_image reference(s)")
print(visitimage_refs)

Found 1 visit_image reference(s)
[DatasetRef(DatasetType('visit_image', {band, instrument, day_obs, detector, physical_filter, visit}, ExposureF), {instrument: 'LSSTComCam', detector: 0, visit: 2024120200214, band: 'r', day_obs: 20241202, physical_filter: 'r_03'}, run='LSSTComCam/runs/DRP/DP1/DM-53601', id=019b1949-bf82-75d8-9788-83e1233c1d1f)]


In [6]:
# Step 2: Extract visit and detector, retrieve the visit_image
visit_id = visitimage_refs[0].dataId['visit']
detector_id = visitimage_refs[0].dataId['detector']
print(f"Visit: {visit_id}, Detector: {detector_id}")

visit_image = butler.get(visitimage_refs[0])
psf = visit_image.getPsf()
print(f"PSF retrieved. Postage stamp size: {psf.computeImage(lsst.geom.Point2D(300,300)).getDimensions()} pix")

Visit: 2024120200214, Detector: 0
PSF retrieved. Postage stamp size: (25, 25) pix


In [7]:
# Step 3: Query dp1.Source for stars used in PSF modeling (calib_psf_used = 1)
query = (
    "SELECT sourceId, x, y "
    "FROM dp1.Source "
    f"WHERE visit = {visit_id} "
    f"AND detector = {detector_id} "
    "AND calib_psf_used = 1"
)

job = rsp_tap.submit_job(query)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase is', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()

psf_used_table = job.fetch_result().to_table()
print(f"Found {len(psf_used_table)} PSF candidate stars")

Job phase is COMPLETED
Found 287 PSF candidate stars


In [ ]:
import numpy as np

def radial_profile(image, center=None):
    """Compute the azimuthally averaged radial profile of a 2D image."""
    image = np.asarray(image, dtype=float)
    ny, nx = image.shape

    if center is None:
        # First-pass center assumption; this can be replaced later
        # by a centroid-based or PSF-model-based center definition.
        center = ((nx - 1) / 2.0, (ny - 1) / 2.0)

    x0, y0 = center
    y, x = np.indices(image.shape)
    r = np.sqrt((x - x0) ** 2 + (y - y0) ** 2)
    r_bin = np.floor(r).astype(int)

    sum_per_bin = np.bincount(r_bin.ravel(), weights=image.ravel())
    count_per_bin = np.bincount(r_bin.ravel())
    radius_per_bin = np.bincount(r_bin.ravel(), weights=r.ravel())

    valid = count_per_bin > 0
    profile = sum_per_bin[valid] / count_per_bin[valid]
    radius = radius_per_bin[valid] / count_per_bin[valid]

    return radius, profile


In [ ]:
# Step 4: Plot PSF vs Double Gaussian fit for a selection of stars (paginated)

# Define double Gaussian function (radial profile)
# Matches the LSST DoubleGaussianPsf model: f(r) = A1*exp(-r^2/2*sigma1^2) + A2*exp(-r^2/2*sigma2^2)
# See: https://pipelines.lsst.io/py-api/lsst.meas.algorithms.DoubleGaussianPsf.html
def double_gaussian(r, a1, sigma1, a2, sigma2):
    g1 = a1 * np.exp(-r**2 / (2.0 * sigma1**2))
    g2 = a2 * np.exp(-r**2 / (2.0 * sigma2**2))
    return g1 + g2

# --- COMPUTATION PHASE ---
n_stars = len(psf_used_table)
SIGMA_TO_FWHM = 2.0 * np.sqrt(2.0 * np.log(2.0))

t0 = time.time()
results = []
n_skipped = 0
for i in range(n_stars):
    try:
        x_star = psf_used_table['x'][i]
        y_star = psf_used_table['y'][i]
        position_star = lsst.geom.Point2D(x_star, y_star)

        psf_model = psf.computeImage(position_star)
        psf_array = psf_model.array
        xlen, ylen = psf_model.getDimensions()

        psf_shape = psf.computeShape(position_star)
        psf_sigma = psf_shape.getDeterminantRadius()
        psf_fwhm = psf_sigma * SIGMA_TO_FWHM

        yy, xx = np.indices((ylen, xlen))
        distances = np.sqrt((xx - xlen / 2.0)**2 + (yy - ylen / 2.0)**2)

        r_flat = distances.ravel()
        psf_flat = psf_array.ravel()

        p0 = [np.max(psf_flat), psf_sigma,
              np.max(psf_flat) * 0.1, psf_sigma * 2.0]
        bounds_low = [0, 0.1, 0, 0.1]
        bounds_high = [np.inf, xlen, np.inf, xlen]
        try:
            pars, _ = curve_fit(double_gaussian, r_flat, psf_flat,
                                p0=p0, bounds=(bounds_low, bounds_high),
                                maxfev=5000)
        except RuntimeError:
            pars = p0

        dgauss_array = double_gaussian(distances, *pars).reshape(psf_array.shape)
        residual = psf_array - dgauss_array

        center = ((xlen - 1) / 2.0, (ylen - 1) / 2.0)
        rp_radius, rp_psf = radial_profile(psf_array, center=center)
        _, rp_dg = radial_profile(dgauss_array, center=center)
        rp_residual = rp_psf - rp_dg
        max_val = np.max(np.abs(psf_array))

        results.append({
            'x': x_star, 'y': y_star,
            'psf_array': psf_array,
            'dgauss_array': dgauss_array,
            'residual': residual,
            'max_val': max_val,
            'fwhm': psf_fwhm,
            'pars': pars,
            'rp_radius': rp_radius,
            'rp_psf': rp_psf,
            'rp_dg': rp_dg,
            'rp_residual': rp_residual,
        })

    except Exception as e:
        n_skipped += 1
        print("Star %d skipped: %s" % (i, e))

t_compute = time.time() - t0
print("Computation done: %d/%d stars in %.2fs (%d skipped)" % (
    len(results), n_stars, t_compute, n_skipped))

# --- VISUALIZATION PHASE ---
page_size = 10
n_computed = len(results)

def plot_page(results, start, page_size, visit_id, detector_id, my_band):
    end = min(start + page_size, len(results))
    n = end - start
    if n <= 0:
        return
    fig, axes = plt.subplots(n, 5, figsize=(24, 5 * n))
    if n == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle(
        "PSF vs Double Gaussian - Visit %s, Detector %s, Band %s (stars %d-%d of %d)"
        % (visit_id, detector_id, my_band, start + 1, end, len(results)),
        fontsize=14, y=1.01
    )

    for idx, i in enumerate(range(start, end)):
        r = results[i]
        max_val = r['max_val']

        title_psf = "Star %d: PSF Model (PIFF)\n(x=%.0f, y=%.0f)\nFWHM=%.2f pix" % (
            i + 1, r['x'], r['y'], r['fwhm'])
        im0 = axes[idx, 0].imshow(r['psf_array'], vmin=-max_val, vmax=max_val,
                                  cmap='viridis', origin='lower')
        axes[idx, 0].set_title(title_psf, fontsize=10)
        fig.colorbar(im0, ax=axes[idx, 0], shrink=0.8)

        title_fit = "Double Gaussian Fit\nA1=%.4f, s1=%.2f\nA2=%.4f, s2=%.2f" % (
            r['pars'][0], r['pars'][1], r['pars'][2], r['pars'][3])
        im1 = axes[idx, 1].imshow(r['dgauss_array'], vmin=-max_val, vmax=max_val,
                                  cmap='viridis', origin='lower')
        axes[idx, 1].set_title(title_fit, fontsize=10)
        fig.colorbar(im1, ax=axes[idx, 1], shrink=0.8)

        res_scale = max_val / 10
        im2 = axes[idx, 2].imshow(r['residual'], vmin=-res_scale, vmax=res_scale,
                                  cmap='RdBu_r', origin='lower')
        axes[idx, 2].set_title("Residual\n(PSF - Double Gaussian)", fontsize=10)
        fig.colorbar(im2, ax=axes[idx, 2], shrink=0.8)

        axes[idx, 3].plot(r['rp_radius'], r['rp_psf'], 'o-', ms=3, lw=1.2, label='PSF')
        axes[idx, 3].plot(r['rp_radius'], r['rp_dg'], 's--', ms=3, lw=1.2, label='Double Gaussian')
        axes[idx, 3].set_title('Radial Profile', fontsize=10)
        axes[idx, 3].set_xlabel('Radius [pixel]')
        axes[idx, 3].set_ylabel('Mean intensity')
        axes[idx, 3].grid(alpha=0.3)
        axes[idx, 3].legend(fontsize=8)

        axes[idx, 4].axhline(0.0, color='k', lw=1)
        axes[idx, 4].plot(r['rp_radius'], r['rp_residual'], 'o-', ms=3, lw=1.2, color='tab:red')
        axes[idx, 4].set_title('Profile Residual', fontsize=10)
        axes[idx, 4].set_xlabel('Radius [pixel]')
        axes[idx, 4].set_ylabel('PSF - Double Gaussian')
        axes[idx, 4].grid(alpha=0.3)

        for ax in axes[idx, :3]:
            ax.set_xlabel('x (pixel)')
            ax.set_ylabel('y (pixel)')

    plt.tight_layout()
    plt.show()

t0 = time.time()
for page_start in range(0, n_computed, page_size):
    plot_page(results, page_start, page_size, visit_id, detector_id, my_band)
t_plot = time.time() - t0
print("Plotting done: %d stars in %.2fs (%d per page)" % (n_computed, t_plot, page_size))